In [13]:
import pandas
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification,BertTokenizer, BertForSequenceClassification
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding

dataset_path = r"../../datasets/sentiment_dataset.csv"

In [14]:
dataset = pandas.read_csv(dataset_path)

In [15]:
train_df, test_df = train_test_split(dataset, test_size=0.2, random_state=42, stratify=dataset["label"])

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

In [16]:
train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 3525
})

In [17]:
checkpoint = "distilbert-base-uncased"
checkpoint = "vinai/phobert-base" 
## https://huggingface.co/vinai/phobert-base 
# Lưu ý: vinai/phobert-base phải max_length = 256 

In [18]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
# Ensure consistent label names for training + deploy
model.config.id2label = {0: 'NEGATIVE', 1: 'POSITIVE'}
model.config.label2id = {'NEGATIVE': 0, 'POSITIVE': 1}
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: a1c4912d-8c80-43e3-b82c-63a783cd1341)')' thrown while requesting HEAD https://huggingface.co/vinai/phobert-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at vinai/phobert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [19]:
# PhoBERT (RoBERTa-style) requires fixed max_length to avoid shape mismatches.
max_length = 256

def tokenizer_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False,
        return_token_type_ids=False,
    )


def preprocessing(ds):
    ds = ds.map(tokenizer_fn, batched=True, remove_columns=["text"])
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
    return ds

In [20]:
print(train_dataset['text'][0])

 Đến lần muốn đến thêm lần ngon tuyệt


In [21]:
tokenized_train = preprocessing(train_dataset)
tokenized_test = preprocessing(test_dataset)

Map: 100%|██████████| 882/882 [00:00<00:00, 2163.34 examples/s]


In [22]:
import numpy as np
import torch
from transformers import TrainingArguments, Trainer

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds)
        , 'f1': f1_score(labels, preds)
    }

training_args = TrainingArguments(
    output_dir='../../models/save/bert-finetuned-negative-positive',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    # warmup_ratio=0.06,
    warmup_steps = int(0.06 * len(tokenized_train) // 16),  # 6% of training steps
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    disable_tqdm=True,
    logging_steps=50,
    seed=42,
    report_to='none',
)

from transformers import ProgressCallback

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    # tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[ProgressCallback()],
)


In [26]:
train_result = trainer.train()
metrics = trainer.evaluate()
print('Trainer metrics:', metrics)

# Detailed report on the test split
import json
from pathlib import Path
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

pred_output = trainer.predict(tokenized_test)
y_pred = np.argmax(pred_output.predictions, axis=-1)
y_true = pred_output.label_ids

target_names = [model.config.id2label[i] for i in sorted(model.config.id2label.keys())]
report_dict = classification_report(
    y_true,
    y_pred,
    target_names=target_names,
    digits=4,
    output_dict=True,
)
cm = confusion_matrix(y_true, y_pred)

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=target_names, digits=4))
print('Confusion matrix:')
print(cm)

# Save metrics to output/metrics.json (repo root/output)
repo_root = Path.cwd().resolve().parents[1]  # .../source/finetune -> repo root
out_path = repo_root / 'output' / 'metrics.json'
out_path.parent.mkdir(parents=True, exist_ok=True)

payload = {
    'checkpoint': checkpoint,
    'max_length': max_length,
    'trainer_metrics': {k: float(v) for k, v in metrics.items() if isinstance(v, (int, float))},
    'classification_report': report_dict,
    'confusion_matrix': cm.tolist(),
}
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print(f"\nSaved metrics to: {out_path}")

  8%|▊         | 51/663 [00:10<02:03,  4.94it/s]

{'loss': 0.1439, 'grad_norm': 22.590639114379883, 'learning_rate': 1.8892307692307693e-05, 'epoch': 0.23}
{'loss': 0.1439, 'grad_norm': 22.590639114379883, 'learning_rate': 1.8892307692307693e-05, 'epoch': 0.22624434389140272}


 15%|█▌        | 101/663 [00:20<01:49,  5.14it/s]

{'loss': 0.1464, 'grad_norm': 0.12412509322166443, 'learning_rate': 1.7353846153846156e-05, 'epoch': 0.45}
{'loss': 0.1464, 'grad_norm': 0.12412509322166443, 'learning_rate': 1.7353846153846156e-05, 'epoch': 0.45248868778280543}


 23%|██▎       | 151/663 [00:29<01:48,  4.74it/s]

...

{'train_runtime': 157.2365, 'train_samples_per_second': 67.255, 'train_steps_per_second': 4.217, 'train_loss': 0.09817511042745765, 'epoch': 3.0}
{'train_runtime': 157.2365, 'train_samples_per_second': 67.255, 'train_steps_per_second': 4.217, 'train_loss': 0.09817511042745765, 'epoch': 3.0}


100%|██████████| 28/28 [00:02<00:00, 11.29it/s]


{'eval_loss': 0.2104158103466034, 'eval_accuracy': 0.9455782312925171, 'eval_f1': 0.9548872180451128, 'eval_runtime': 2.767, 'eval_samples_per_second': 318.751, 'eval_steps_per_second': 10.119, 'epoch': 3.0}
Trainer metrics: {'eval_loss': 0.2104158103466034, 'eval_accuracy': 0.9455782312925171, 'eval_f1': 0.9548872180451128, 'eval_runtime': 2.767, 'eval_samples_per_second': 318.751, 'eval_steps_per_second': 10.119, 'epoch': 3.0}


100%|██████████| 28/28 [00:02<00:00, 11.35it/s]


Classification report:
              precision    recall  f1-score   support

    NEGATIVE     0.9395    0.9235    0.9314       353
    POSITIVE     0.9495    0.9603    0.9549       529

    accuracy                         0.9456       882
   macro avg     0.9445    0.9419    0.9432       882
weighted avg     0.9455    0.9456    0.9455       882

Confusion matrix:
[[326  27]
 [ 21 508]]

Saved metrics to: F:\AI Project\bert-finetuned-positive-negative-comment\output\metrics.json


In [24]:
finetuned_model_path = '../../output/bert-finetuned-negative-positive'

In [25]:
token_save = tokenizer.save_pretrained(finetuned_model_path)
# trained = trainer.save_model("./bert-finetuned-negative-positive")
trained = model.save_pretrained(finetuned_model_path)